  # Whisper Accent Robustness — Model Performance Evaluation







  Run `eval_model_perf.py` on SLURM first to generate the CSVs this notebook reads.







  - **Combined** → scripted + spontaneous test data (single CSV per model)

  - **Per-domain** → filter by `domain` column within each CSV







  Metrics:



  - **WER**  — word error rate (primary ASR metric)

  - **PER**  — phoneme error rate via G2P; labelled "PER (G2P)" throughout

In [1]:
# ── Config ────────────────────────────────────────────────────────────────────
RESULTS_DIR = "results/model_perf_comparison"


# Keys must match {model_key} in CSV filenames; values are display labels
MODEL_KEYS = {
    "baseline": "Whisper ZS",
    "whisper_finetuned": "Whisper FT",
    # "whisper_finetuned_hoc": "Whisper FT (heldout Chinese)",
    "whisper_finetuned_hov": "Whisper FT (heldout Vietnamese)",
    "whisfusion": "WhisFusion ZS",
    "whisfusion_ft": "WhisFusion FT",
    # "whisfusion_ft_hoc": "WhisFusion (heldout Chinese)",
    "whisfusion_ft_hov": "WhisFusion (heldout Vietnamese)",
}
DOMAINS = ["scripted", "spontaneous"]  # Filter options within combined CSVs




In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
import jiwer

pd.set_option("display.max_colwidth", 80)




In [3]:
def load_results(model_key: str) -> pd.DataFrame | None:
    """Load combined predictions CSV (scripted + spontaneous)."""
    p = Path(RESULTS_DIR) / f"{model_key}_predictions.csv"
    if not p.exists():
        print(f"  [missing] {p} — run eval_model_perf.py first")
        return None
    return pd.read_csv(p)


def filter_by_domain(df: pd.DataFrame, domain: str | None) -> pd.DataFrame:
    """Filter combined DataFrame by domain (or return all)."""
    if domain is None:
        return df
    return df[df["domain"] == domain].copy()


# ── Load all cached CSVs ──────────────────────────────────────────────────────
results: dict[str, pd.DataFrame | None] = {}
for key in MODEL_KEYS:
    df = load_results(key)
    results[key] = df
    if df is not None:
        print(f"  loaded  {key}: {len(df):,} rows ({df['domain'].value_counts().to_dict()})")
    else:
        print(f"  [missing] {key}")



  loaded  baseline: 7,796 rows ({'scripted': 7796})
  loaded  whisper_finetuned: 7,796 rows ({'scripted': 7796})
  loaded  whisper_finetuned_hov: 7,796 rows ({'scripted': 7796})
  loaded  whisfusion: 7,796 rows ({'scripted': 7796})
  loaded  whisfusion_ft: 7,796 rows ({'scripted': 7796})
  loaded  whisfusion_ft_hov: 7,796 rows ({'scripted': 7796})


In [4]:
# ── Helpers ──────────────────────────────────────────────────────────────────


def available(key: str) -> bool:
    return results.get(key) is not None


def get_data(key: str, domain: str | None = None) -> pd.DataFrame | None:
    """Get data for model, optionally filtered by domain."""
    df = results.get(key)
    if df is None:
        return None
    return filter_by_domain(df, domain)


def corpus_wer(df: pd.DataFrame) -> float:
    return float(jiwer.wer(
        df["reference_norm"].fillna("").tolist(),
        df["prediction_norm"].fillna("").tolist(),
    ))

def corpus_mer(df: pd.DataFrame) -> float:
    return float(jiwer.mer(
        df["reference_norm"].fillna("").tolist(),
        df["prediction_norm"].fillna("").tolist(),
    ))

def corpus_per(df: pd.DataFrame) -> float | None:
    """Mean utterance PER (G2P-derived)."""
    if "utt_per" not in df.columns:
        return None
    vals = df["utt_per"].dropna()
    return float(vals.mean()) if len(vals) else None


def grouped_wer(df: pd.DataFrame, group_col: str = "l1") -> pd.DataFrame:
    rows = []
    for grp, sub in df.groupby(group_col):
        rows.append({
            group_col:  grp,
            "num_utts": len(sub),
            "wer":      float(jiwer.wer(
                                sub["reference_norm"].fillna("").tolist(),
                                sub["prediction_norm"].fillna("").tolist(),
                            )),
            "mer":      float(jiwer.mer(
                                sub["reference_norm"].fillna("").tolist(),
                                sub["prediction_norm"].fillna("").tolist(),
                            )),
            "per":      float(sub["utt_per"].dropna().mean())
                        if "utt_per" in sub.columns else None,
        })
    return pd.DataFrame(rows)


print("Helpers loaded.")



Helpers loaded.


  ---



  # Part 1 — Overall WER & PER (G2P)



  **Default: Combined (all data).** Use `domain="scripted"` or `"spontaneous"` in `get_summary_table()` for domain-specific views.

In [5]:
def get_summary_table(domain: str | None = None) -> pd.DataFrame:
    """WER/PER table for domain (None = all data)."""
    rows = []
    for key, label in MODEL_KEYS.items():
        df = get_data(key, domain)
        if df is None:
            continue
        wer = corpus_wer(df)
        per = corpus_per(df)
        mer = corpus_mer(df)
        rows.append({"Model": f"{label}", "WER": wer, "MER": mer, "PER (G2P)": per})
    return pd.DataFrame(rows)


domain = None
df = get_summary_table(domain)
display(
    df.style
    .format({
        "WER": "{:.4f}",
        "MER": "{:.4f}",
        "PER (G2P)": lambda v: f"{v:.4f}" if pd.notna(v) else "—",
    })
    .background_gradient(cmap="RdYlGn_r", subset=["WER", "MER", "PER (G2P)"])
    .set_caption("WER & PER (G2P) — All Data (Scripted + Spontaneous)")
)


,Model,WER,MER,PER (G2P)
0,Whisper ZS,0.1475,0.1455,0.0766
1,Whisper FT,0.0653,0.0648,0.0394
2,Whisper FT (heldout Vietnamese),0.0783,0.0777,0.0464
3,WhisFusion ZS,0.2530,0.2470,0.1407
4,WhisFusion FT,0.0833,0.0823,0.0647
5,WhisFusion (heldout Vietnamese),0.1059,0.1042,0.0818


In [6]:
from plotly.subplots import make_subplots


def plot_summary_bars(domain: str | None = None):
    """Plot WER/PER bars for domain (None = all data)."""
    sub = get_summary_table(domain)
    if sub.empty:
        return

    domain_str = f" ({domain})" if domain else " (All Data)"

    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=("WER", "MER", "PER (G2P)"),
        vertical_spacing=0.12,
    )

    x = sub["Model"].tolist()

    for row, col_name, label in [
        (1, "WER", "WER"),
        (2, "MER", "MER"),
        (3, "PER (G2P)", "PER (G2P)"),
    ]:
        vals = sub[col_name]
        if col_name == "PER (G2P)" and not vals.notna().any():
            continue

        fig.add_trace(
            go.Bar(
                name=label,
                x=x,
                y=vals.tolist(),
                text=[f"{v:.1%}" if pd.notna(v) else "" for v in vals],
                textposition="outside",
                showlegend=False,
                cliponaxis=False,
            ),
            row=row, col=1
        )

    fig.update_yaxes(title_text="Error Rate", tickformat=".0%", row=1, col=1)
    fig.update_yaxes(tickformat=".0%", row=2, col=1)
    fig.update_yaxes(tickformat=".0%", row=3, col=1)

    fig.update_xaxes(tickangle=-20, automargin=True)

    fig.update_layout(
        title=f"WER vs PER by Model{domain_str}",
        height=950,
        width=1200,
        margin=dict(t=100, b=80, l=60, r=40),
        bargap=0.35,
    )
    fig.show()

domain = "scripted"
plot_summary_bars(domain)


  ---



  # Part 2 — Per-L1 WER



  **Default: Combined (all data).** Use `domain=` for domain-specific L1 breakdowns.

In [7]:
def get_l1_breakdown(domain: str | None = None, metric: str = "wer") -> pd.DataFrame:
    """Per-L1 breakdown for a chosen metric."""
    metric = metric.lower()
    if metric not in {"wer", "mer", "per"}:
        raise ValueError("metric must be one of: 'wer', 'mer', 'per'")

    l1_rows = []
    for key, label in MODEL_KEYS.items():
        df = get_data(key, domain)
        if df is None:
            continue
        grp = grouped_wer(df, "l1")
        grp["Model"] = label
        l1_rows.append(grp)

    if not l1_rows:
        return pd.DataFrame()

    l1_df = pd.concat(l1_rows, ignore_index=True)

    # Delta vs zero-shot baseline (negative = improvement)
    if "baseline" in MODEL_KEYS and available("baseline"):
        base_label = MODEL_KEYS["baseline"]
        base_col = f"{metric}_base"
        base_vals = (
            l1_df[l1_df["Model"] == base_label][["l1", metric]]
            .rename(columns={metric: base_col})
        )
        l1_df = l1_df.merge(base_vals, on="l1", how="left")
        l1_df[f"{metric}_delta_pct"] = (
            (l1_df[metric] - l1_df[base_col]) / l1_df[base_col] * 100
        )

    return l1_df


domain = "scripted"   # None / "scripted" / "spontaneous"
metric = "wer"  # "wer" / "mer" / "per"

l1_df = get_l1_breakdown(domain, metric)

if not l1_df.empty:
    l1s = sorted(l1_df["l1"].unique())
    metric_label = "PER (G2P)" if metric == "per" else metric.upper()

    # Use actual model keys that exist in MODEL_KEYS and are available
    model_order = list(MODEL_KEYS.keys())
    ordered_items = [(k, MODEL_KEYS[k]) for k in model_order if available(k)]

    fig = go.Figure()
    for key, label in ordered_items:
        sub = l1_df[l1_df["Model"] == label].set_index("l1")
        fig.add_trace(go.Bar(
            name=label,
            x=l1s,
            y=[sub.loc[l, metric] if l in sub.index else None for l in l1s],
            text=[f"{sub.loc[l, metric]:.1%}" if l in sub.index and pd.notna(sub.loc[l, metric]) else "" for l in l1s],
            textposition="outside",
        ))

    domain_str = f" (domain={domain})" if domain else " (All Data)"
    fig.update_layout(
        title=f"{metric_label} by L1{domain_str}",
        barmode="group",
        yaxis=dict(title=metric_label, tickformat=".0%"),
        legend=dict(orientation="h", y=1.12, xanchor="center", x=0.5),
        margin=dict(t=80),
    )
    fig.show()

    out = Path(RESULTS_DIR) / f"comparison_{domain or 'all'}_{metric}_by_l1.csv"
    l1_df.to_csv(out, index=False)
    print(f"Saved {out}")

    display(
        l1_df.sort_values(["l1", "Model"])
            .style.format({
                metric: "{:.4f}",
                f"{metric}_base": "{:.4f}",
                f"{metric}_delta_pct": "{:+.1f}%",
                "per": lambda v: f"{v:.4f}" if pd.notna(v) else "—",
            })
            .set_caption(f"{domain_str} — Per-L1 {metric_label}")
    )


Saved results/model_perf_comparison/comparison_scripted_wer_by_l1.csv


,l1,num_utts,wer,mer,per,Model,wer_base,wer_delta_pct
35,Arabic,1132,0.0735,0.072772,0.0577,WhisFusion (heldout Vietnamese),0.1268,-42.0%
28,Arabic,1132,0.0563,0.055819,0.0441,WhisFusion FT,0.1268,-55.6%
21,Arabic,1132,0.2412,0.235488,0.1243,WhisFusion ZS,0.1268,+90.3%
7,Arabic,1132,0.0451,0.044893,0.0261,Whisper FT,0.1268,-64.4%
14,Arabic,1132,0.0499,0.049644,0.0286,Whisper FT (heldout Vietnamese),0.1268,-60.6%
0,Arabic,1132,0.1268,0.125196,0.0578,Whisper ZS,0.1268,+0.0%
36,Chinese,1130,0.1165,0.113630,0.0892,WhisFusion (heldout Vietnamese),0.1916,-39.2%
29,Chinese,1130,0.0969,0.095285,0.0737,WhisFusion FT,0.1916,-49.4%
22,Chinese,1130,0.3210,0.311768,0.1850,WhisFusion ZS,0.1916,+67.6%
8,Chinese,1130,0.0927,0.091897,0.0569,Whisper FT,0.1916,-51.6%


  ---



  # Part 3 — Domain Gap Analysis (zero-shot)



  Scripted vs spontaneous **within** each L1, using baseline model.

In [8]:
# def plot_domain_gap(model_key: str = "baseline"):
#     """Scripted vs spontaneous gap by L1."""
#     df = results.get(model_key)
#     if df is None:
#         return
    
#     s_g  = grouped_wer(df[df["domain"] == "scripted"], "l1").rename(columns={"wer": "WER_scripted"})
#     sp_g = grouped_wer(df[df["domain"] == "spontaneous"], "l1").rename(columns={"wer": "WER_spontaneous"})
    
#     gap = s_g[["l1", "WER_scripted"]].merge(
#         sp_g[["l1", "WER_spontaneous"]], on="l1", how="inner")
#     gap["gap"] = gap["WER_spontaneous"] - gap["WER_scripted"]
    
#     if gap.empty:
#         print(f"No domain gap data for {model_key}")
#         return
    
#     l1s = gap["l1"].tolist()
#     fig = go.Figure()
#     fig.add_trace(go.Bar(name="Scripted", x=l1s, y=gap["WER_scripted"].tolist()))
#     fig.add_trace(go.Bar(name="Spontaneous", x=l1s, y=gap["WER_spontaneous"].tolist()))
#     fig.update_layout(
#         title=f"{MODEL_KEYS.get(model_key, model_key)} — Scripted vs Spontaneous by L1",
#         barmode="group",
#         yaxis=dict(title="WER", tickformat=".0%"),
#         legend=dict(orientation="h", y=1.12, xanchor="center", x=0.5),
#     )
#     fig.show()
#     display(
#         gap.style
#          .format({c: "{:.4f}" for c in ["WER_scripted", "WER_spontaneous", "gap"]})
#          .set_caption(f"{MODEL_KEYS.get(model_key, model_key)} — Domain Gap by L1")
#     )


# plot_domain_gap("baseline")



  ---



  # Part 4 — Utterance-level Analysis



  **Default: Combined**. Filter by `domain=` for specific views.

In [9]:
import seaborn as sns

def plot_utt_metric_dist_sns(domain: str | None = None, metric: str = "wer", bw_adjust: float = 1.0):
    import matplotlib.pyplot as plt

    metric_map = {
        "wer": ("utt_wer", "WER"),
        "mer": ("utt_mer", "MER"),
        "per": ("utt_per", "PER (G2P)"),
    }

    models = ["whisper_finetuned_hoc", "whisfusion_ft_hoc"]

    metric = metric.lower()
    if domain is not None and domain not in DOMAINS:
        raise ValueError(f"domain must be one of {DOMAINS} or None")
    if metric not in metric_map:
        raise ValueError("metric must be one of: 'wer', 'mer', 'per'")

    metric_col, metric_label = metric_map[metric]

    plt.figure(figsize=(11, 5.5))
    n_traces = 0
    for key, label in [(k, MODEL_KEYS[k]) for k in models if available(k)]:
        df = get_data(key, domain)
        df = df[df["l1"] == "Arabic"]

        if df is None or metric_col not in df.columns:
            continue
        vals = pd.to_numeric(df[metric_col], errors="coerce").dropna()
        if vals.empty:
            continue
        sns.kdeplot(vals, bw_adjust=bw_adjust, label=label, linewidth=1)
        n_traces += 1

    if n_traces == 0:
        print(f"No data available for metric='{metric}' and domain='{domain}'.")
        return

    domain_str = f" (domain={domain})" if domain else " (All Data)"
    plt.title(f"{metric_label} Density{domain_str}")
    plt.xlabel(metric_label)
    plt.ylabel("Density")
    plt.legend(title=None, loc="upper center", bbox_to_anchor=(0.5, 1.12), ncol=min(4, n_traces))
    plt.tight_layout()
    plt.show()

# usage
plot_utt_metric_dist_sns(domain=None, metric="mer", bw_adjust=1.0)


No data available for metric='mer' and domain='None'.


<Figure size 1100x550 with 0 Axes>

  ---



  # Part 5 — Worst Utterances (cross-model)



  **Default: Combined**. Use `domain=` for filtering.

In [10]:
# ── Worst utterances per model — cross-model comparison ───────────────────────
N_WORST = 30
models = [
    "baseline",
    "whisper_finetuned",
    "whisfusion_ft",
]
domain = "scripted"  # None = all data; "scripted"; "spontaneous"
exclude_l1s = []  # e.g. ["Jamaican"]; use [] to keep all L1s
metric = "mer"  # "wer" / "mer" / "per"

if metric not in {"wer", "mer", "per"}:
    raise ValueError("metric must be one of: 'wer', 'mer', 'per'")
if domain is not None and domain not in DOMAINS:
    raise ValueError(f"domain must be one of {DOMAINS} or None")

metric_col = f"utt_{metric}"
join_keys = ["utterance_id", "speaker", "l1", "domain"]

base_key = models[0]
base_df = get_data(base_key, domain)
if base_df is None:
    raise ValueError(f"Missing data for {base_key}")

for anchor_key in models:
    if not available(anchor_key):
        continue

    anchor_df = get_data(anchor_key, domain)
    if anchor_df is None:
        continue

    if exclude_l1s:
        anchor_df = anchor_df[~anchor_df["l1"].isin(exclude_l1s)].copy()
        base_df_f = base_df[~base_df["l1"].isin(exclude_l1s)].copy()
    else:
        base_df_f = base_df

    if metric_col not in anchor_df.columns:
        print(f"Skipping {anchor_key}: missing {metric_col}")
        continue

    # Select top-N worst ROWS (not just utterance_id, which is non-unique)
    worst_anchor = (
        anchor_df.nlargest(N_WORST, metric_col)[join_keys]
        .drop_duplicates()
        .head(N_WORST)
        .copy()
    )

    # Start from baseline reference rows on exact keys
    worst = worst_anchor.merge(
        base_df_f[join_keys + ["reference_norm"]],
        on=join_keys,
        how="left",
    )

    for key in models:
        if not available(key):
            continue

        other = get_data(key, domain)
        if other is None:
            continue
        if exclude_l1s:
            other = other[~other["l1"].isin(exclude_l1s)].copy()
        if metric_col not in other.columns:
            continue

        add = (
            other[join_keys + ["prediction_norm", metric_col]]
            .rename(columns={
                "prediction_norm": f"pred_{key}",
                metric_col: f"{metric}_{key}",
            })
        )
        worst = worst.merge(add, on=join_keys, how="left")

    domain_str = f" (domain={domain})" if domain else " (All Data)"
    exclude_str = f" | excluded L1s: {', '.join(exclude_l1s)}" if exclude_l1s else ""
    metric_label = metric.upper()
    fmt_cols = {c: "{:.3f}" for c in worst.columns if c.startswith(f"{metric}_")}

    display(
        worst.style
        .format(fmt_cols)
        .set_properties(**{
            "max-width": "300px",
            "white-space": "normal",
            "overflow-wrap": "break-word",
            "word-break": "break-word",
            "overflow": "visible",
        })
        .set_caption(
            f"Top-{len(worst)} Worst Utterances by {metric_label} for {MODEL_KEYS[anchor_key]}{domain_str}{exclude_str}"
        )
    )


,utterance_id,speaker,l1,domain,reference_norm,pred_baseline,mer_baseline,pred_whisper_finetuned,mer_whisper_finetuned,pred_whisfusion_ft,mer_whisfusion_ft
0,arctic_a0015,BWC,Chinese,scripted,its the aurora borealis,is a rural place,1.000,its a rule of belief,0.800,its the aur bore,0.500
1,arctic_b0240,BWC,Chinese,scripted,there are four all low mcoy answered,therefore allow my call answer,1.000,there are four all along mcoy answered,0.143,there are four all my answered,0.286
2,arctic_b0377,BWC,Chinese,scripted,a scarlet loincloth,and scarlett linkles,1.000,and scarlett linked to me,1.000,and scarlet lo loco,0.750
3,arctic_a0263,EBVS,Spanish,scripted,joan looked triumphantly at sheldon who bowed,join lookitrumpfully and shut down cooperate,1.000,joan looked triumphantly and shied off coo,0.571,joan looked looked triumphantly and sheldeded,0.625
4,arctic_a0346,EBVS,Spanish,scripted,get down and dig in,good town auntie king,1.000,get down antique hand,0.600,but but in in,0.800
5,arctic_a0480,EBVS,Spanish,scripted,tom spink has a harpoon,dont speak her phone,1.000,tonspink has harpoon,0.600,sp spink has a haron,0.400
6,arctic_b0262,EBVS,Spanish,scripted,saxons onto her job,sections on doha yop,1.000,saxons on her yob,0.500,saxons her her job,0.250
7,arctic_a0015,HQTV,Vietnamese,scripted,its the aurora borealis,is it already boring,1.000,its an aurora borealis,0.250,its our aurora borealis,0.250
8,arctic_a0134,HQTV,Vietnamese,scripted,he obeyed the pressure of her hand,hes all made of fresh uphill ahead,1.000,hell all be the pressure of his hand,0.500,heeded the pressure of him,0.571
9,arctic_a0155,HQTV,Vietnamese,scripted,wont you draw up gentlemen,one youve rarred and the other youve chandlemen,1.000,wont you draw up gentlemen,0.000,won you draw in gentlemen gentlemen,0.500


,utterance_id,speaker,l1,domain,reference_norm,pred_baseline,mer_baseline,pred_whisper_finetuned,mer_whisper_finetuned,pred_whisfusion_ft,mer_whisfusion_ft
0,arctic_b0377,BWC,Chinese,scripted,a scarlet loincloth,and scarlett linkles,1.000,and scarlett linked to me,1.000,and scarlet lo loco,0.750
1,arctic_a0296,HQTV,Vietnamese,scripted,bassett was a fastidious man,possessed were a fatidious man,0.600,passed were the fatidious men,1.000,bassness was a fid man man,0.500
2,arctic_a0309,HQTV,Vietnamese,scripted,white leghorns said missus mortimer,while redhorn says mrs mortimer,0.800,why lets have such mischievous more timber,1.000,why ishorn said missusus mortimer,0.600
3,arctic_b0220,HQTV,Vietnamese,scripted,harry bancroft dave lied,every bankrupt they fly,1.000,every bone crops they fly,1.000,every bancroft lied,0.500
4,bdl_arctic_a0484,BDL,English,scripted,nosiree,no surrey,1.000,no sir ee,1.000,no sir ee,1.000
5,arctic_a0296,ZHAA,Arabic,scripted,bassett was a fastidious man,pass it well as a fastidious man,0.571,pass it well if i asked you as man,0.889,bassett was a fastidous man,0.200
6,arctic_b0376,EBVS,Spanish,scripted,thought i and a worthy fool he proved,so i am awarded for his proofs,0.875,thorpe i am awarded for his proof,0.875,th i i and a worthy for he proved,0.333
7,arctic_b0200,HQTV,Vietnamese,scripted,we leave the eventuality to time and law,he lived the eventuality to time and law,0.250,he lived there eventually to torment loh,0.875,he left the eventuality to time and law,0.250
8,arctic_a0534,BWC,Chinese,scripted,but johannes could and did,by joining us good and dead,0.833,patrick and i were good and dead,0.857,but johannes and and did,0.200
9,arctic_a0534,HQTV,Vietnamese,scripted,but johannes could and did,but your handy good and did,0.500,what your handy good and deed,0.833,but johannes could could did did,0.333


,utterance_id,speaker,l1,domain,reference_norm,pred_baseline,mer_baseline,pred_whisper_finetuned,mer_whisper_finetuned,pred_whisfusion_ft,mer_whisfusion_ft
0,bdl_arctic_a0484,BDL,English,scripted,nosiree,no surrey,1.000,no sir ee,1.000,no sir ee,1.000
1,arctic_a0272,EBVS,Spanish,scripted,you can take a vacation on pay,you can take a vacation with me,0.286,you can take a vacation on pay,0.000,donon take the canation was,0.857
2,arctic_b0467,HQTV,Vietnamese,scripted,let us run them for ourselves,lets add random for our service,1.000,let us run down for our service,0.429,let is domed for our works,0.833
3,arctic_a0346,EBVS,Spanish,scripted,get down and dig in,good town auntie king,1.000,get down antique hand,0.600,but but in in,0.800
4,arctic_b0377,BWC,Chinese,scripted,a scarlet loincloth,and scarlett linkles,1.000,and scarlett linked to me,1.000,and scarlet lo loco,0.750
5,arctic_b0351,EBVS,Spanish,scripted,matthewson whos this bookkeeper rogers,mattsonson who is this bookkeeperrogers,0.833,matthewson who is this bookkeeper rogers,0.333,matthewson who who is this bookkeeperkeeperers,0.714
6,arctic_a0309,HJK,Korean,scripted,white leghorns said missus mortimer,white legrines that misses more to more,0.857,white leghorns said missus mortimer,0.000,white leghorns miss miss mort mort more,0.714
7,arctic_b0240,HQTV,Vietnamese,scripted,there are four all low mcoy answered,therefore only one more answer,1.000,therefore only low mcoy answered,0.571,there is four lowy,0.714
8,arctic_a0016,EBVS,Spanish,scripted,theres fort churchill a rifle shot beyond the ridge asleep,there is 4th churchill arrived for chang with jando ridge as asleep,0.750,there is fort churchill a rifle found beyond the ridge as it leapt,0.462,theres churchill to a chle from r r as asleep,0.700
9,arctic_a0277,EBVS,Spanish,scripted,mcoy found a stifling poisonous atmosphere in the pent cabin,marco found a stifling postsynos atmosphere in the pentcaving,0.400,mahcoy found a stifling post synonymous atmosphere in the pent cabin,0.273,my found a a stiflingous thegroundous the pentess,0.700


In [11]:
def compare_models_head_to_head(
    model_key1: str,
    model_key2: str,
    domain: str | None = None,
    metric: str = "wer",
    top_n: int = 5,
):
    """Compare two models: wins, losses, draws, and examples with predictions."""
    metric = metric.lower()
    if metric not in {"wer", "mer", "per"}:
        raise ValueError("metric must be one of: 'wer', 'mer', 'per'")

    metric_col = f"utt_{metric}"
    df1 = get_data(model_key1, domain)
    df2 = get_data(model_key2, domain)

    if df1 is None or df2 is None:
        print("Missing data for one or both models")
        return

    if metric_col not in df1.columns or metric_col not in df2.columns:
        print(f"Missing {metric_col} in one or both models")
        return

    join_keys = ["utterance_id", "speaker", "l1", "domain"]

    merged = (
        df1[join_keys + ["reference_norm", "prediction_norm", metric_col]]
        .rename(columns={
            "prediction_norm": f"pred_{model_key1}",
            metric_col: f"{metric}_1",
        })
        .merge(
            df2[join_keys + ["prediction_norm", metric_col]]
            .rename(columns={
                "prediction_norm": f"pred_{model_key2}",
                metric_col: f"{metric}_2",
            }),
            on=join_keys,
            how="inner",
        )
    )

    merged["result"] = merged.apply(
        lambda r: "win" if r[f"{metric}_1"] < r[f"{metric}_2"]
        else ("loss" if r[f"{metric}_1"] > r[f"{metric}_2"] else "draw"),
        axis=1,
    )

    wins = int((merged["result"] == "win").sum())
    losses = int((merged["result"] == "loss").sum())
    draws = int((merged["result"] == "draw").sum())
    total = len(merged)

    model1_label = MODEL_KEYS.get(model_key1, model_key1)
    model2_label = MODEL_KEYS.get(model_key2, model_key2)
    domain_str = f" ({domain})" if domain else " (All Data)"

    print(f"\n{'='*60}")
    print(f"{model1_label} vs {model2_label}{domain_str}")
    print(f"Metric: {metric.upper()}")
    print(f"{'='*60}")
    print(f"Wins (lower {metric.upper()}):    {wins:5d} ({wins/total*100:5.1f}%)")
    print(f"Losses (higher {metric.upper()}):  {losses:5d} ({losses/total*100:5.1f}%)")
    print(f"Draws (equal):     {draws:5d} ({draws/total*100:5.1f}%)")
    print(f"Total utterances:  {total:5d}")
    print(f"{'='*60}\n")

    wins_df = (
        merged[merged["result"] == "win"]
        .nlargest(top_n, f"{metric}_2")
        .loc[:, [
            "utterance_id",
            "speaker",
            "l1",
            "domain",
            "reference_norm",
            f"pred_{model_key1}",
            f"pred_{model_key2}",
            f"{metric}_1",
            f"{metric}_2",
        ]]
        .rename(columns={
            f"pred_{model_key1}": model1_label,
            f"pred_{model_key2}": model2_label,
            f"{metric}_1": f"{model1_label} {metric.upper()}",
            f"{metric}_2": f"{model2_label} {metric.upper()}",
        })
    )

    print(f"Top {top_n} Wins for {model1_label}:")
    display(
        wins_df.style.format({
            f"{model1_label} {metric.upper()}": "{:.3f}",
            f"{model2_label} {metric.upper()}": "{:.3f}",
        })
    )

# Example usage:
compare_models_head_to_head(
    "whisfusion_ft",
    "whisper_finetuned",
    domain="scripted",
    metric="mer",
    top_n=20,
)


WhisFusion FT vs Whisper FT (scripted)
Metric: MER
Wins (lower MER):     1218 ( 15.6%)
Losses (higher MER):   2091 ( 26.8%)
Draws (equal):      4487 ( 57.6%)
Total utterances:   7796

Top 20 Wins for WhisFusion FT:


,utterance_id,speaker,l1,domain,reference_norm,WhisFusion FT,Whisper FT,WhisFusion FT MER,Whisper FT MER
967,arctic_b0377,BWC,Chinese,scripted,a scarlet loincloth,and scarlet lo loco,and scarlett linked to me,0.750,1.000
3563,arctic_a0296,HQTV,Vietnamese,scripted,bassett was a fastidious man,bassness was a fid man man,passed were the fatidious men,0.500,1.000
3576,arctic_a0309,HQTV,Vietnamese,scripted,white leghorns said missus mortimer,why ishorn said missusus mortimer,why lets have such mischievous more timber,0.600,1.000
4080,arctic_b0220,HQTV,Vietnamese,scripted,harry bancroft dave lied,every bancroft lied,every bone crops they fly,0.500,1.000
5827,arctic_a0296,ZHAA,Arabic,scripted,bassett was a fastidious man,bassett was a fastidous man,pass it well if i asked you as man,0.200,0.889
2097,arctic_b0376,EBVS,Spanish,scripted,thought i and a worthy fool he proved,th i i and a worthy for he proved,thorpe i am awarded for his proof,0.333,0.875
4060,arctic_b0200,HQTV,Vietnamese,scripted,we leave the eventuality to time and law,he left the eventuality to time and law,he lived there eventually to torment loh,0.250,0.875
533,arctic_a0534,BWC,Chinese,scripted,but johannes could and did,but johannes and and did,patrick and i were good and dead,0.200,0.857
3801,arctic_a0534,HQTV,Vietnamese,scripted,but johannes could and did,but johannes could could did did,what your handy good and deed,0.333,0.833
3896,arctic_b0036,HQTV,Vietnamese,scripted,he wondered too where roscoe was,he wanteded two where roscoe,he wanted to wear crosscoals,0.500,0.833
